## Prompt Chaining in LangGraph

Prompt chaining is a workflow pattern in which a task is broken into a sequence of smaller LLM calls, where the output of one call is fed as input into the next. Instead of asking a single prompt to produce a complex, multi-part result in one shot, the problem is decomposed into ordered stages, each handled by its own prompt and each building on the result of the stage before it.

**When to use prompt chaining instead of a single LLM call**

A single LLM call works well when the task is simple enough that the model can reliably produce the full desired output in one pass. Prompt chaining becomes useful when:

- The task naturally decomposes into distinct sub-tasks that each benefit from a focused prompt (for example, first planning a structure, then writing content that follows that structure).
- A single monolithic prompt would ask the model to do too much at once, increasing the risk of the model skipping steps, losing structure, or producing shallower output.
- Downstream stages depend on a well-formed intermediate artifact from an earlier stage (an outline, a plan, an extracted set of facts) rather than needing to be inferred implicitly inside one long prompt.
- Debuggability matters: the developer wants to inspect, log, or reuse the intermediate outputs, not just the final answer.

**How LangGraph implements this as a sequential graph**

In LangGraph, prompt chaining is implemented as a `StateGraph` with multiple nodes, each node wrapping one LLM call, wired together linearly with `add_edge`. Every node receives the shared state, reads the fields it needs, calls the model, writes its output back into the state, and returns the updated state. Because each node's output is written into the same shared state object, and each subsequent node reads from that same state, data flows forward through the chain: node 2 can see what node 1 produced, node 3 can see what nodes 1 and 2 produced, and so on. The graph itself only encodes control flow (which node runs after which); the actual data dependency between stages is expressed through the state fields the nodes choose to read and write. This notebook builds one such chain: a node that turns a blog title into an outline, followed by a node that turns that outline into full blog content.

**Tradeoffs**

- **Latency**: Each additional node is a separate LLM call, executed one after another. Total latency is roughly the sum of the latencies of every node in the chain, which is higher than a single call, though each individual call can be faster since it has a narrower job.
- **Decomposition quality**: Splitting a task into stages generally improves the reliability and structure of the final output, because each prompt has a single, well-defined responsibility instead of juggling multiple concerns simultaneously.
- **Intermediate state inspection and debuggability**: Because intermediate results (like the outline in this notebook) are stored as explicit fields in the shared state, they can be printed, logged, or inspected independently of the final output, which makes it much easier to diagnose where a chain produced a bad result.
- **Rigidity**: A plain sequential chain (nodes linked purely with `add_edge`) always executes every stage in the same fixed order for every input. It has no branching or conditional logic, so it cannot skip stages, retry a stage, or take a different path based on intermediate results — capabilities that require the more complex, non-linear graph structures covered later.

### Imports

`StateGraph` is the core class used to define a LangGraph workflow as a graph of nodes and edges. `START` and `END` are sentinel markers used to wire the entry point and exit point of the graph. `ChatOpenAI` is the chat model wrapper used to call an OpenAI LLM from each node. `TypedDict` is used to declare the shape of the graph's state as a typed dictionary. `load_dotenv` loads environment variables (such as the OpenAI API key) from a `.env` file so the model client can authenticate.

In [1]:
from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI
from typing import TypedDict
from dotenv import load_dotenv

### Environment and Model Setup

`load_dotenv()` reads the `.env` file in the project and loads its variables (such as `OPENAI_API_KEY`) into the process environment, which `ChatOpenAI` needs to authenticate its API calls. `model = ChatOpenAI()` instantiates the chat model client that every node in this notebook will call with `model.invoke(prompt)`. Because the same `model` object is reused across all nodes, both stages of the chain (outline generation and blog writing) go through the same underlying LLM configuration.

In [2]:
load_dotenv()
model = ChatOpenAI()

### State Schema: `BlogState`

`BlogState` is a `TypedDict` that defines the shared state passed between every node in the graph. It has three fields, each corresponding to one stage of the chain:

- `title`: the input to the workflow, the blog topic supplied by the caller when the graph is invoked. It is set once, before the graph runs, and is read by both nodes.
- `outline`: an intermediate artifact, populated by the `create_outline` node. It exists so that the outline-generation step and the content-writing step can be separated into distinct LLM calls, with the outline persisted in state rather than being recomputed or implicitly re-derived inside a single prompt.
- `content`: the final output of the workflow, populated by the `create_blog` node using both `title` and `outline`.

Because all three fields live in one `TypedDict`, the state accumulates over the run: it starts with only `title`, gains `outline` after the first node runs, and gains `content` after the second node runs. Each node reads whichever fields it needs and adds its own field to the same dictionary before returning it.

In [3]:
class BlogState(TypedDict):

    title: str
    outline: str
    content: str

### Node 1: `create_outline`

This is the first stage of the chain. It reads `title` from the incoming state, builds the prompt `"Generate a detailed outline for a blog on the topic - {title}"`, and calls `model.invoke(prompt).content` to get the LLM's response text. The resulting outline is written into `state['outline']`, and the full state dictionary is returned. Its sole responsibility is turning a topic into a structured outline; it does not attempt to write the blog itself. This keeps the prompt narrow and focused, which is the core idea of prompt chaining: instead of asking the model to plan and write a blog in one call, planning is isolated into its own step whose output becomes the well-defined input for the next step.

In [4]:
def create_outline(state: BlogState) -> BlogState:

    # fetch title
    title = state['title']

    # call llm gen outline
    prompt = f'Generate a detailed outline for a blog on the topic - {title}'
    outline = model.invoke(prompt).content

    # update state
    state['outline'] = outline

    return state

### Node 2: `create_blog`

This is the second stage of the chain, and it demonstrates the defining property of prompt chaining: it reads both `title` and `outline` from state, meaning it depends directly on the output that `create_outline` produced in the previous stage. Its prompt, `"Write a detailed blog on the title - {title} using the follwing outline \n {outline}"`, explicitly injects the prior node's output back into a new LLM call. The model's response is written into `state['content']`, and the updated state is returned. Because this node only runs after `create_outline` has already populated `outline`, the graph's edge ordering (defined in the next cell) must guarantee that `create_outline` executes first — the chain's control flow and its data flow are aligned by construction.

In [5]:
def create_blog(state: BlogState) -> BlogState:

    title = state['title']
    outline = state['outline']

    prompt = f'Write a detailed blog on the title - {title} using the follwing outline \n {outline}'

    content = model.invoke(prompt).content

    state['content'] = content

    return state

### Building and Compiling the Graph

`graph = StateGraph(BlogState)` creates a new graph whose shared state must conform to the `BlogState` schema. The two functions are registered as nodes with `graph.add_node('create_outline', create_outline)` and `graph.add_node('create_blog', create_blog)`, binding each string name to its corresponding node function.

The edges then define the strictly linear execution order that makes this a sequential chain rather than an arbitrary graph:

- `graph.add_edge(START, 'create_outline')`: execution begins at `create_outline`.
- `graph.add_edge('create_outline', 'create_blog')`: once `create_outline` finishes, control passes to `create_blog`. This single edge is what guarantees the outline exists in state before the blog-writing prompt is built.
- `graph.add_edge('create_blog', END)`: once `create_blog` finishes, the graph terminates.

`workflow = graph.compile()` compiles this node/edge definition into a runnable object (`workflow`) that can be invoked with an initial state.

In [6]:
graph = StateGraph(BlogState)

# nodes
graph.add_node('create_outline', create_outline)
graph.add_node('create_blog', create_blog)

# edges
graph.add_edge(START, 'create_outline')
graph.add_edge('create_outline', 'create_blog')
graph.add_edge('create_blog', END)

workflow = graph.compile()

### Invoking the Workflow

`intial_state = {'title': 'Rise of AI in India'}` supplies only the one field the chain needs as input; `outline` and `content` are absent at this point and will be added as the graph runs. `final_state = workflow.invoke(intial_state)` runs the compiled graph start to finish: `create_outline` executes first, populating `outline`, and its returned state is passed as input to `create_blog`, which populates `content`. `workflow.invoke` returns the final accumulated state after `END` is reached, so `final_state` is a single dictionary containing all three fields — `title`, `outline`, and `content` — printed in the cell output. The `LangSmith` error messages in the output are unrelated telemetry warnings from a background tracing call and do not affect the workflow's result.

In [7]:
intial_state = {'title': 'Rise of AI in India'}
final_state = workflow.invoke(intial_state)

print(final_state)

Failed to multipart ingest runs: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')
Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


{'title': 'Rise of AI in India', 'outline': "I. Introduction\n    A. Brief overview of Artificial Intelligence (AI) \n    B. Explanation of why AI is becoming increasingly prevalent in India\n    C. Thesis statement: The rise of AI in India is revolutionizing various industries and shaping the future of the country\n\nII. Historical Background of AI in India\n    A. Introduction of AI technologies in India\n    B. Growth of AI research and development in the country\n    C. Impact of government policies and initiatives on the adoption of AI in India\n\nIII. Key Industries Adopting AI in India\n    A. Healthcare sector\n        1. Use of AI in medical diagnosis and treatment\n        2. AI-powered healthcare solutions improving patient care\n    B. Banking and finance industry\n        1. AI applications in fraud detection and risk assessment \n        2. Chatbots and virtual assistants enhancing customer service\n    C. Education sector\n        1. Personalized learning experiences thr

Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


### Inspecting the Intermediate Output

`print(final_state['outline'])` isolates and displays only the outline produced by the first node in the chain. This is only possible because `outline` was retained as its own explicit field in `BlogState` rather than being an ephemeral value discarded after being consumed by `create_blog`. Being able to inspect this intermediate artifact independently of the final blog content is one of the practical debuggability benefits of prompt chaining described in the introduction: if the final content were poor, this cell would let a developer check whether the problem originated in the planning stage (a bad outline) or the writing stage (a bad blog built from a good outline).

In [8]:
print(final_state['outline'])

I. Introduction
    A. Brief overview of Artificial Intelligence (AI) 
    B. Explanation of why AI is becoming increasingly prevalent in India
    C. Thesis statement: The rise of AI in India is revolutionizing various industries and shaping the future of the country

II. Historical Background of AI in India
    A. Introduction of AI technologies in India
    B. Growth of AI research and development in the country
    C. Impact of government policies and initiatives on the adoption of AI in India

III. Key Industries Adopting AI in India
    A. Healthcare sector
        1. Use of AI in medical diagnosis and treatment
        2. AI-powered healthcare solutions improving patient care
    B. Banking and finance industry
        1. AI applications in fraud detection and risk assessment 
        2. Chatbots and virtual assistants enhancing customer service
    C. Education sector
        1. Personalized learning experiences through AI
        2. Adoption of AI in administrative tasks and s

### Inspecting the Final Output

`print(final_state['content'])` displays the final output field, the blog article produced by `create_blog`. Read alongside the outline printed in the previous cell, this makes it possible to verify that the second stage actually followed the structure produced by the first stage, confirming that the two-stage chain functioned as intended: topic to outline, outline to full content.

In [9]:
print(final_state['content'])

Artificial Intelligence (AI) has been a game-changer in various industries across the globe, and India is no exception. The rise of AI in India is revolutionizing the way businesses operate, the way people access healthcare services, and the way students learn. With the increasing adoption of AI technologies in the country, India is poised to become a major player in the global AI landscape.

Historical Background of AI in India

India has a long history of embracing technology and innovation. The introduction of AI technologies in the country can be traced back to the early 2000s when companies started experimenting with machine learning algorithms and predictive analytics. Over the years, the growth of AI research and development in India has been fueled by the availability of skilled professionals, a thriving startup ecosystem, and government policies that promote innovation and technology adoption.

The government of India has also played a significant role in promoting the adoptio

## Summary

This notebook implements prompt chaining as a two-node linear `StateGraph`: `create_outline` turns a `title` into an `outline`, and `create_blog` turns that `outline` (together with the original `title`) into `content`. The nodes are wired with `add_edge` in a single fixed sequence — `START` to `create_outline` to `create_blog` to `END` — so every invocation follows the exact same path, and each node's output is exposed as a distinct field in `BlogState`, which is what makes the intermediate outline separately inspectable from the final content.

This fixed, single-path structure is the simplest way to compose multiple LLM calls in LangGraph, and it is the foundation the rest of this folder builds on. Later notebooks move beyond a strictly linear chain: branching workflows use conditional edges so that the graph can route to different nodes depending on the content of the state rather than always following the same next step, and parallelization workflows run multiple nodes concurrently from a shared point in the graph and then combine their results, rather than forcing every stage to wait on exactly one predecessor. Both of those patterns still rely on the same underlying mechanics demonstrated here — a shared, accumulating state object read and written by node functions — but relax the assumption that the graph is a single straight line from `START` to `END`.